# Single-example final-layer Q/K geometry around predicted line breaks

This notebook samples one deterministic example and inspects the activation immediately before the final attention layer.

1. It plots a 3D PCA trajectory of the pre-final-attention activation at every token position, labelled by the cumulative character count at that position.
2. For every position whose target is `_NEWLINE_`—that is, every position where the model is asked to predict a line break—it fits a separate joint 3D PCA to:
   - the final-layer Head 3 query `W_Q` applied to the current activation; and
   - Head 3 keys `W_K` applied to every previous-position activation.

The query/key projection includes the final block's `ln1`, matching the model's actual attention computation. Every plot includes the origin and vector segments from the origin. The notebook also plots the Head 3 `W_Q` embedding at every token position, color-coded by character count, plus a joint plot of all `W_Q` and `W_K` embeddings and their pairwise inner-product heatmap and the corresponding unnormalized `exp(inner product)` heatmap, the normalized causal attention heatmap, and attention reconstructed from inner products in the global 3D PCA subspace. Set `USE_ACTIVATION_PCA_FOR_ATTENTION` to `True` to use the already-computed initial activation-trajectory PCA coordinates for every attention geometry plot. When it is `False`, every attention plot uses the one global joint `W_Q`/`W_K` PCA from the all-embeddings plot; the actual attention weights remain unchanged. Set `QUERY_POSITIONS` to `"newline"` (default), `"all"`, or an explicit list of positions; each selected query displays its Q/K geometry, its actual Head 3 attention pattern, a 3D-PCA-reconstructed attention pattern, and the raw 3D-PCA QK inner products.

In [61]:
from pathlib import Path
import json
import math

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from IPython.display import display
from sklearn.decomposition import PCA

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
RANDOM_SEED = 19_001  # Change this to inspect a different sampled example.
EXAMPLE_INDEX = None  # None samples one example from the seeded batch.
HEAD = 3              # Zero-based head index.
QUERY_POSITIONS = [55,56,57,58,59,60,61]  # "newline", "all", or a list of zero-based positions.
USE_ACTIVATION_PCA_FOR_ATTENTION = False  # True: use initial activation PCA coordinates.
COLLECTION_BATCH_SIZE = 128


In [62]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
final_layer = n_layers - 1
activation_layer = final_layer - 1
n_heads = cfg["model"]["n_heads"]
if activation_layer < 0:
    raise ValueError("The model must have at least two transformer layers.")
if not 0 <= HEAD < n_heads:
    raise ValueError(f"HEAD must be between 0 and {n_heads - 1}.")
ACTIVATION_SITE = f"blocks.{activation_layer}.hook_resid_post"
PATTERN_SITE = f"blocks.{final_layer}.attn.hook_pattern"
PATTERN_SITE = f"blocks.{final_layer}.attn.hook_pattern"
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Pre-final activation site: {ACTIVATION_SITE}")
print(f"Final attention layer: {final_layer}, head: {HEAD}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Pre-final activation site: blocks.1.hook_resid_post
Final attention layer: 2, head: 3


## Sample one datapoint

In [63]:
# Generate a reproducible batch, then choose one example from it. Set
# EXAMPLE_INDEX to an integer to inspect a specific member of that batch.
generator = torch.Generator(device="cpu").manual_seed(RANDOM_SEED)
batch = project_data.make_batch(
    batch_size=COLLECTION_BATCH_SIZE,
    vocab=vocab,
    task_cfg=cfg["task"],
    device=device,
    seed=RANDOM_SEED,
)
if EXAMPLE_INDEX is None:
    example_index = int(torch.randint(COLLECTION_BATCH_SIZE, (), generator=generator).item())
else:
    example_index = int(EXAMPLE_INDEX)
if not 0 <= example_index < COLLECTION_BATCH_SIZE:
    raise ValueError(f"EXAMPLE_INDEX must be between 0 and {COLLECTION_BATCH_SIZE - 1}.")

_, cache = model.run_with_cache(
    batch.tokens,
    prepend_bos=False,
    names_filter=lambda name: name in {ACTIVATION_SITE, PATTERN_SITE},
)
activations = cache[ACTIVATION_SITE][example_index].detach().cpu().float().numpy()
attention_pattern = cache[PATTERN_SITE][example_index, HEAD].detach().cpu().float().numpy()
tokens = batch.tokens[example_index].detach().cpu().tolist()
targets = batch.targets[example_index].detach().cpu().tolist()
line_width = int(batch.line_lengths[example_index].item())

character_counts = []
running_count = 0
for token_id in tokens:
    if token_id == vocab.newline_id:
        running_count = 0
    else:
        running_count += project_data.extract_character_count(vocab.decode_token(token_id))
    character_counts.append(running_count)

token_names = [vocab.decode_token(token_id) for token_id in tokens]
target_names = [vocab.decode_token(token_id) for token_id in targets]
newline_query_positions = [i for i, target_id in enumerate(targets) if target_id == vocab.newline_id]

if QUERY_POSITIONS == "newline":
    query_positions = newline_query_positions
elif QUERY_POSITIONS == "all":
    query_positions = list(range(len(tokens)))
elif isinstance(QUERY_POSITIONS, (list, tuple, np.ndarray)):
    query_positions = [int(position) for position in QUERY_POSITIONS]
else:
    raise ValueError('QUERY_POSITIONS must be "newline", "all", or a list of positions.')
if any(position < 0 or position >= len(tokens) for position in query_positions):
    raise ValueError("QUERY_POSITIONS contains a position outside the sequence.")
query_positions = [position for position in query_positions if position > 0]

print(f"Seed: {RANDOM_SEED}; example index: {example_index}; sampled line width: {line_width}")
print(f"Token positions: {len(tokens)}; selected query positions: {query_positions}")
for i, (token_name, target_name, count) in enumerate(zip(token_names, target_names, character_counts)):
    marker = "  <-- predicts newline" if i in newline_query_positions else ""
    selected = "  [selected query]" if i in query_positions else ""
    print(f"{i:>3}: {token_name:>12}  count={count:>3}  target={target_name}{marker}{selected}")


Seed: 19001; example index: 78; sampled line width: 59
Token positions: 150; selected query positions: [55, 56, 57, 58, 59, 60, 61]
  0:          BOS  count=  0  target=TOKEN_1_3
  1:    TOKEN_1_3  count=  3  target=TOKEN_4_2
  2:    TOKEN_4_2  count=  5  target=TOKEN_9_3
  3:    TOKEN_9_3  count=  8  target=TOKEN_5_1
  4:    TOKEN_5_1  count=  9  target=TOKEN_1_4
  5:    TOKEN_1_4  count= 13  target=TOKEN_5_4
  6:    TOKEN_5_4  count= 17  target=TOKEN_5_3
  7:    TOKEN_5_3  count= 20  target=TOKEN_2_3
  8:    TOKEN_2_3  count= 23  target=TOKEN_5_4
  9:    TOKEN_5_4  count= 27  target=TOKEN_5_1
 10:    TOKEN_5_1  count= 28  target=TOKEN_5_4
 11:    TOKEN_5_4  count= 32  target=TOKEN_0_7
 12:    TOKEN_0_7  count= 39  target=TOKEN_4_3
 13:    TOKEN_4_3  count= 42  target=TOKEN_3_8
 14:    TOKEN_3_8  count= 50  target=TOKEN_7_2
 15:    TOKEN_7_2  count= 52  target=TOKEN_4_7
 16:    TOKEN_4_7  count= 59  target=_NEWLINE_  <-- predicts newline
 17:    _NEWLINE_  count=  0  target=TOKEN_8_2


## PCA of the pre-final-attention activation at every token position

In [64]:
activation_pca = PCA(n_components=3)
activation_coordinates = activation_pca.fit_transform(activations)

def project_head_vectors_for_plot(vectors, projection):
    vectors = torch.as_tensor(vectors, device=device, dtype=model.W_Q.dtype)
    normalized = model.blocks[final_layer].ln1(vectors)
    if projection == "query":
        projected = normalized @ model.W_Q[final_layer, HEAD] + model.b_Q[final_layer, HEAD]
    else:
        projected = normalized @ model.W_K[final_layer, HEAD] + model.b_K[final_layer, HEAD]
    return projected.detach().cpu().float().numpy()

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", marker=dict(size=7, color="black"), name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>"))
fig.add_trace(go.Scatter3d(
    x=activation_coordinates[:, 0], y=activation_coordinates[:, 1], z=activation_coordinates[:, 2],
    mode="lines+markers", marker=dict(size=4, color=character_counts, colorscale="Viridis", colorbar=dict(title="Character count")),
    line=dict(color="rgba(80,80,80,0.6)", width=3), name="Pre-final activation",
    text=[f"{i}: {name}" for i, name in enumerate(token_names)], customdata=np.asarray(character_counts),
    hovertemplate="position=%{text}<br>character count=%{customdata}<extra>Activation</extra>",
))
segments=[]
for point in activation_coordinates:
    segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])
fig.add_trace(go.Scatter3d(x=[p[0] for p in segments], y=[p[1] for p in segments], z=[p[2] for p in segments], mode="lines", line=dict(color="rgba(100,100,100,0.18)", width=1), name="Activation vectors", showlegend=False, hoverinfo="skip"))
fig.update_layout(title=f"Example {example_index}: pre-final-attention activation trajectory<br>PCA variance explained = {activation_pca.explained_variance_ratio_.sum():.1%}", scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"), height=800, margin=dict(l=0, r=0, b=0, t=90))
display(fig)

q_embeddings = project_head_vectors_for_plot(activations, "query")
q_pca = PCA(n_components=3)
q_coordinates = q_pca.fit_transform(q_embeddings)
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", marker=dict(size=7, color="black"), name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>"))
q_segments=[]
for point in q_coordinates:
    q_segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])
fig.add_trace(go.Scatter3d(x=[p[0] for p in q_segments], y=[p[1] for p in q_segments], z=[p[2] for p in q_segments], mode="lines", line=dict(color="rgba(65,105,225,0.18)", width=1), name="W_Q vectors", showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter3d(
    x=q_coordinates[:, 0], y=q_coordinates[:, 1], z=q_coordinates[:, 2], mode="lines+markers",
    marker=dict(size=5, color=character_counts, colorscale="Viridis", colorbar=dict(title="Character count")),
    line=dict(color="royalblue", width=3), name="W_Q embedding", text=[f"{i}: {name}" for i, name in enumerate(token_names)],
    customdata=np.asarray(character_counts), hovertemplate="position=%{text}<br>character count=%{customdata}<extra>W_Q embedding</extra>",
))
fig.update_layout(title=f"Example {example_index}: final-layer Head {HEAD} W_Q embedding by token position<br>PCA variance explained = {q_pca.explained_variance_ratio_.sum():.1%}", scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"), height=800, margin=dict(l=0, r=0, b=0, t=90))
display(fig)

# Joint PCA of all W_Q and W_K embeddings across token positions.
k_embeddings = project_head_vectors_for_plot(activations, "key")
qk_pca = PCA(n_components=3)
qk_coordinates = qk_pca.fit_transform(np.concatenate([q_embeddings, k_embeddings], axis=0))
q_all_coordinates = qk_coordinates[:len(tokens)]
k_all_coordinates = qk_coordinates[len(tokens):]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers", marker=dict(size=7, color="black"),
    name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>",
))
for coordinates, label, color, line_color in [
    (q_all_coordinates, "W_Q embeddings", "royalblue", "rgba(65,105,225,0.18)"),
    (k_all_coordinates, "W_K embeddings", "crimson", "rgba(220,20,60,0.18)"),
]:
    segments = []
    for point in coordinates:
        segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])
    fig.add_trace(go.Scatter3d(
        x=[p[0] for p in segments], y=[p[1] for p in segments], z=[p[2] for p in segments],
        mode="lines", line=dict(color=line_color, width=1),
        name=f"{label} vectors", showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter3d(
        x=coordinates[:, 0], y=coordinates[:, 1], z=coordinates[:, 2],
        mode="lines+markers", marker=dict(size=5, color=character_counts, colorscale="Viridis",
        colorbar=dict(title="Character count")), line=dict(color=color, width=3),
        name=label, text=[f"{i}: {name}" for i, name in enumerate(token_names)],
        customdata=np.asarray(character_counts),
        hovertemplate="position=%{text}<br>character count=%{customdata}<extra>" + label + "</extra>",
    ))
fig.update_layout(
    title=f"Example {example_index}: all Head {HEAD} W_Q and W_K embeddings<br>"
          f"Joint PCA variance explained = {qk_pca.explained_variance_ratio_.sum():.1%}",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=800, margin=dict(l=0, r=0, b=0, t=90),
)
display(fig)

# Pairwise raw inner products between every W_Q and W_K embedding.
qk_inner_products = q_embeddings @ k_embeddings.T
position_labels = [f"{i}: {name}" for i, name in enumerate(token_names)]
fig = go.Figure(data=go.Heatmap(
    z=qk_inner_products,
    x=position_labels,
    y=position_labels,
    colorscale="RdBu_r",
    zmid=0,
    colorbar=dict(title="W_Q · W_K"),
    customdata=np.asarray(character_counts)[:, None, None].repeat(len(tokens), axis=1),
    hovertemplate=(
        "query=%{y}<br>key=%{x}<br>query char count=%{customdata[0]}<br>"
        "inner product=%{z:.4f}<extra></extra>"
    ),
))
fig.update_layout(
    title=f"Example {example_index}: Head {HEAD} pairwise W_Q · W_K inner products",
    xaxis_title="Key position / W_K embedding",
    yaxis_title="Query position / W_Q embedding",
    height=850, margin=dict(l=120, r=40, b=140, t=90),
)
display(fig)

# Unnormalized exponential score e^(W_Q · W_K), shown below the inner-product plot.
# Clipping only protects the visualization from numerical overflow.
exp_qk_scores = np.exp(np.clip(qk_inner_products, -50, 50))
fig = go.Figure(data=go.Heatmap(
    z=exp_qk_scores,
    x=position_labels,
    y=position_labels,
    colorscale="Viridis",
    colorbar=dict(title="exp(W_Q · W_K)"),
    customdata=np.asarray(character_counts)[:, None, None].repeat(len(tokens), axis=1),
    hovertemplate=(
        "query=%{y}<br>key=%{x}<br>query char count=%{customdata[0]}<br>"
        "exp(inner product)=%{z:.4f}<extra></extra>"
    ),
))
fig.update_layout(
    title=f"Example {example_index}: Head {HEAD} unnormalized exponential QK score",
    xaxis_title="Key position / W_K embedding",
    yaxis_title="Query position / W_Q embedding",
    height=850, margin=dict(l=120, r=40, b=140, t=90),
)
display(fig)

# Normalized causal attention weights: the actual row-wise softmax pattern
# cached from the final attention layer for Head 3.
normalized_attention = attention_pattern.copy()
normalized_attention[np.triu_indices(len(tokens), k=1)] = np.nan
fig = go.Figure(data=go.Heatmap(
    z=normalized_attention,
    x=position_labels,
    y=position_labels,
    colorscale="Viridis",
    zmin=0,
    zmax=float(np.nanmax(normalized_attention)),
    colorbar=dict(title="Softmax attention"),
    customdata=np.asarray(character_counts)[:, None, None].repeat(len(tokens), axis=1),
    hovertemplate=(
        "query=%{y}<br>key=%{x}<br>query char count=%{customdata[0]}<br>"
        "normalized attention=%{z:.4f}<extra></extra>"
    ),
))
fig.update_layout(
    title=f"Example {example_index}: Head {HEAD} normalized causal attention",
    xaxis_title="Key position / W_K embedding",
    yaxis_title="Query position / W_Q embedding",
    height=850, margin=dict(l=120, r=40, b=140, t=90),
)
display(fig)

# Attention reconstructed only from the global 3D W_Q/W_K PCA subspace.
# This uses scaled 3D inner products, causal masking, and row-wise softmax.
qk_pca_logits = (q_all_coordinates @ k_all_coordinates.T) / np.sqrt(3.0)
causal_logits = np.where(
    np.tri(len(tokens), len(tokens), dtype=bool),
    qk_pca_logits,
    -np.inf,
)
row_max = np.max(causal_logits, axis=1, keepdims=True)
qk_pca_attention = np.exp(causal_logits - row_max)
qk_pca_attention /= qk_pca_attention.sum(axis=1, keepdims=True)
qk_pca_attention[np.triu_indices(len(tokens), k=1)] = np.nan

fig = go.Figure(data=go.Heatmap(
    z=qk_pca_attention,
    x=position_labels,
    y=position_labels,
    colorscale="Viridis",
    zmin=0,
    zmax=float(np.nanmax(qk_pca_attention)),
    colorbar=dict(title="3D-PCA softmax"),
    customdata=np.asarray(character_counts)[:, None, None].repeat(len(tokens), axis=1),
    hovertemplate=(
        "query=%{y}<br>key=%{x}<br>query char count=%{customdata[0]}<br>"
        "3D-PCA attention=%{z:.4f}<extra></extra>"
    ),
))
fig.update_layout(
    title=f"Example {example_index}: attention reconstructed from global 3D W_Q/W_K PCA",
    xaxis_title="Key position / global-PCA W_K coordinate",
    yaxis_title="Query position / global-PCA W_Q coordinate",
    height=850, margin=dict(l=120, r=40, b=140, t=90),
)
display(fig)


## Joint Q/K PCA for every position that predicts a newline

In [65]:
def project_head_vectors(vectors, projection):
    vectors = torch.as_tensor(vectors, device=device, dtype=model.W_Q.dtype)
    normalized = model.blocks[final_layer].ln1(vectors)
    if projection == "query":
        projected = normalized @ model.W_Q[final_layer, HEAD] + model.b_Q[final_layer, HEAD]
    elif projection == "key":
        projected = normalized @ model.W_K[final_layer, HEAD] + model.b_K[final_layer, HEAD]
    else:
        raise ValueError("projection must be 'query' or 'key'.")
    return projected.detach().cpu().float().numpy()

query_records = []
for query_position in query_positions:
    query_vector = project_head_vectors(activations[query_position:query_position + 1], "query")
    previous_vectors = project_head_vectors(activations[:query_position], "key")
    if len(previous_vectors) == 0:
        continue
    all_vectors = np.concatenate([query_vector, previous_vectors], axis=0)
    pca = PCA(n_components=3)
    coordinates = pca.fit_transform(all_vectors)
    query_records.append({
        "query_position": query_position,
        "query_coordinates": coordinates[0],
        "key_coordinates": coordinates[1:],
        "key_positions": list(range(query_position)),
        "pca": pca,
        "attention": attention_pattern[query_position, :query_position],
    })

if not query_records:
    raise RuntimeError("This example has no target-newline positions with previous context.")
print(f"Joint Q/K plots: {len(query_records)}")
for record in query_records:
    print(f"Query position {record['query_position']}: PCA variance explained "
          f"{record['pca'].explained_variance_ratio_.sum():.1%}")


Joint Q/K plots: 7
Query position 55: PCA variance explained 96.5%
Query position 56: PCA variance explained 96.7%
Query position 57: PCA variance explained 97.3%
Query position 58: PCA variance explained 98.0%
Query position 59: PCA variance explained 97.1%
Query position 60: PCA variance explained 96.5%
Query position 61: PCA variance explained 96.6%


In [66]:
if not isinstance(USE_ACTIVATION_PCA_FOR_ATTENTION, (bool, np.bool_)):
    raise TypeError("USE_ACTIVATION_PCA_FOR_ATTENTION must be True or False.")
print(f"Rendering {len(query_records)} attention figures using "
      f"{'activation trajectory PCA' if USE_ACTIVATION_PCA_FOR_ATTENTION else 'global W_Q/W_K PCA'}.")

for record in query_records:
    query_position = record["query_position"]
    if USE_ACTIVATION_PCA_FOR_ATTENTION:
        q = np.asarray(activation_coordinates[query_position], dtype=float)
        k = np.asarray(activation_coordinates[:query_position], dtype=float)
        geometry_label = "activation PCA coordinates"
    else:
        q = np.asarray(q_all_coordinates[query_position], dtype=float)
        k = np.asarray(k_all_coordinates[:query_position], dtype=float)
        geometry_label = "global W_Q/W_K PCA coordinates"
    positions = record["key_positions"]
    origin = np.zeros(3)
    k_segments = []
    for point in k:
        k_segments.extend([origin, point, [None, None, None]])

    # Render the 3D geometry as its own figure. Keeping it separate from the
    # 2D bar chart avoids renderer issues with mixed scene/xy subplots.
    geometry_fig = go.Figure()
    geometry_fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode="markers", marker=dict(size=7, color="black"),
        name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>",
    ))
    geometry_fig.add_trace(go.Scatter3d(
        x=[p[0] for p in k_segments], y=[p[1] for p in k_segments], z=[p[2] for p in k_segments],
        mode="lines", line=dict(color="rgba(220,20,60,0.22)", width=2),
        name="Previous-position K vectors", hoverinfo="skip",
    ))
    geometry_fig.add_trace(go.Scatter3d(
        x=k[:, 0], y=k[:, 1], z=k[:, 2], mode="markers",
        marker=dict(size=4, color=character_counts[:query_position], colorscale="Plasma",
                    colorbar=dict(title="Prior char count")),
        name="K embeddings", text=[f"position {p}: {token_names[p]}" for p in positions],
        customdata=np.asarray([character_counts[p] for p in positions]),
        hovertemplate="%{text}<br>character count=%{customdata}<extra>K embedding</extra>",
    ))
    geometry_fig.add_trace(go.Scatter3d(
        x=[0, q[0], None], y=[0, q[1], None], z=[0, q[2], None],
        mode="lines", line=dict(color="rgba(65,105,225,0.8)", width=6),
        name="Query vector", hoverinfo="skip",
    ))
    geometry_fig.add_trace(go.Scatter3d(
        x=[q[0]], y=[q[1]], z=[q[2]], mode="markers",
        marker=dict(size=9, color="royalblue", symbol="diamond"), name="Q embedding",
        hovertemplate=(f"query position={query_position}<br>"
                       f"character count={character_counts[query_position]}"
                       "<extra>Q embedding</extra>"),
    ))
    geometry_fig.update_layout(
        title=(f"Final layer {final_layer}, head {HEAD}: query position {query_position}<br>"
               f"{geometry_label}"),
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
        height=760, width=1000, margin=dict(l=0, r=0, b=0, t=100),
    )
    display(geometry_fig)

    # Reconstruct attention from the global W_Q/W_K 3D PCA coordinates.
    # This is intentionally based on the W_Q/W_K PCA even when the geometry
    # toggle above is showing residual-activation PCA coordinates.
    q_3d = np.asarray(q_all_coordinates[query_position], dtype=float)
    k_3d = np.asarray(k_all_coordinates[:query_position], dtype=float)
    raw_pca_inner_products = k_3d @ q_3d
    pca_logits = raw_pca_inner_products / np.sqrt(3.0)
    pca_weights = np.exp(pca_logits - np.max(pca_logits))
    pca_weights /= pca_weights.sum()
    actual_weights = np.asarray(record["attention"], dtype=float)

    attention_fig = go.Figure()
    attention_fig.add_trace(go.Bar(
        x=positions, y=actual_weights, name="Actual attention",
        marker_color="crimson",
        customdata=np.asarray([[token_names[p], character_counts[p]] for p in positions], dtype=object),
        hovertemplate=("position=%{x}<br>token=%{customdata[0]}<br>"
                       "character count=%{customdata[1]}<br>attention=%{y:.4f}"
                       "<extra>Actual attention</extra>"),
    ))
    attention_fig.add_trace(go.Bar(
        x=positions, y=pca_weights, name="3D-PCA reconstructed attention",
        marker_color="royalblue",
        customdata=np.asarray([[token_names[p], character_counts[p]] for p in positions], dtype=object),
        hovertemplate=("position=%{x}<br>token=%{customdata[0]}<br>"
                       "character count=%{customdata[1]}<br>3D-PCA attention=%{y:.4f}"
                       "<extra>3D-PCA reconstructed attention</extra>"),
    ))
    attention_fig.update_layout(
        barmode="group",
        title=(f"Query position {query_position}: actual vs. 3D-PCA reconstructed attention"),
        xaxis_title="Previous key position", yaxis_title="Attention weight",
        height=650, width=1200, margin=dict(l=70, r=40, b=100, t=90),
    )
    display(attention_fig)

    raw_inner_product_fig = go.Figure(go.Bar(
        x=positions, y=raw_pca_inner_products,
        marker_color="darkorange", name="Raw 3D-PCA inner product",
        customdata=np.asarray([[token_names[p], character_counts[p]] for p in positions], dtype=object),
        hovertemplate=("position=%{x}<br>token=%{customdata[0]}<br>"
                       "character count=%{customdata[1]}<br>raw inner product=%{y:.4f}"
                       "<extra>Raw 3D-PCA inner product</extra>"),
    ))
    raw_inner_product_fig.update_layout(
        title=f"Query position {query_position}: raw 3D-PCA QK inner products",
        xaxis_title="Previous key position", yaxis_title="Raw W_Q · W_K inner product in 3D PCA space",
        height=550, width=1200, margin=dict(l=80, r=40, b=100, t=90),
    )
    display(raw_inner_product_fig)


Rendering 7 attention figures using global W_Q/W_K PCA.
